# Bayesian Optimization (Using Optuna)
**Method:** Informed Search.
Bayesian optimization builds a probability model of the objective function and uses it to select the most promising hyperparameters to evaluate next.

We use **Optuna**, a modern, highly efficient HPO framework. It uses the Tree-structured Parzen Estimator (TPE) sampler by default.

#### 1) Importing Necessary Libraires:

In [2]:
# To Install Optuna Library:

# !pip install optuna

In [12]:
import optuna
import sklearn.datasets
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

#### 2) Loading Data:

In [4]:
data = sklearn.datasets.load_diabetes()
X, y = data.data, data.target

In [5]:
X

array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
         0.01990749, -0.01764613],
       [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
        -0.06833155, -0.09220405],
       [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
         0.00286131, -0.02593034],
       ...,
       [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
        -0.04688253,  0.01549073],
       [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
         0.04452873, -0.02593034],
       [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
        -0.00422151,  0.00306441]], shape=(442, 10))

In [6]:
y

array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
        69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
        68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
        87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
       259.,  53., 190., 142.,  75., 142., 155., 225.,  59., 104., 182.,
       128.,  52.,  37., 170., 170.,  61., 144.,  52., 128.,  71., 163.,
       150.,  97., 160., 178.,  48., 270., 202., 111.,  85.,  42., 170.,
       200., 252., 113., 143.,  51.,  52., 210.,  65., 141.,  55., 134.,
        42., 111.,  98., 164.,  48.,  96.,  90., 162., 150., 279.,  92.,
        83., 128., 102., 302., 198.,  95.,  53., 134., 144., 232.,  81.,
       104.,  59., 246., 297., 258., 229., 275., 281., 179., 200., 200.,
       173., 180.,  84., 121., 161.,  99., 109., 115., 268., 274., 158.,
       107.,  83., 103., 272.,  85., 280., 336., 281., 118., 317., 235.,
        60., 174., 259., 178., 128.,  96., 126., 28

In [7]:
X.shape

(442, 10)

In [8]:
y.shape

(442,)

#### 3) Defining The Objective Function:

In [9]:
# Optuna will optimize this function. It must return a value to minimize or maximize:

def objective(trial):
    # Suggest values for hyperparameters
    # 'suggest_int', 'suggest_float', 'suggest_categorical' are used to define the search space
    
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    min_samples_split = trial.suggest_float('min_samples_split', 0.1, 1.0)
    min_samples_leaf = trial.suggest_float('min_samples_leaf', 0.1, 0.5)
    
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    
    # Calculate Score using Cross-Validation (Maximize R2 score)
    # n_jobs=-1 uses all cores
    score = cross_val_score(model, X, y, n_jobs=-1, cv=3).mean()
    
    return score

#### 4) Create a Study and Optimize:

In [13]:
# direction='maximize' because we want higher R2 score

study = optuna.create_study(direction='maximize')

[I 2025-12-08 21:22:39,142] A new study created in memory with name: no-name-9ccfb171-3ee6-4e94-b893-20d62e33e8a9


In [15]:
study.optimize(objective, n_trials=20) # Running for 20 trials

[I 2025-12-08 21:22:53,733] Trial 0 finished with value: 0.04662535059096203 and parameters: {'n_estimators': 171, 'max_depth': 12, 'min_samples_split': 0.37509008008414624, 'min_samples_leaf': 0.32609035689667776}. Best is trial 0 with value: 0.04662535059096203.
[I 2025-12-08 21:22:58,669] Trial 1 finished with value: 0.04781894691375551 and parameters: {'n_estimators': 41, 'max_depth': 30, 'min_samples_split': 0.6458821563433426, 'min_samples_leaf': 0.1297454324416385}. Best is trial 1 with value: 0.04781894691375551.
[I 2025-12-08 21:23:03,171] Trial 2 finished with value: -0.0048189261480082735 and parameters: {'n_estimators': 138, 'max_depth': 5, 'min_samples_split': 0.5422795937473355, 'min_samples_leaf': 0.3697332177573788}. Best is trial 1 with value: 0.04781894691375551.
[I 2025-12-08 21:23:07,679] Trial 3 finished with value: -0.004982150048006802 and parameters: {'n_estimators': 167, 'max_depth': 8, 'min_samples_split': 0.827554976052796, 'min_samples_leaf': 0.2851203548848

In [16]:
# Results:

print("\n--- Results ---")
print(f"Best Value (R2): {study.best_value:.4f}")
print(f"Best Hyperparameters: {study.best_params}")


--- Results ---
Best Value (R2): 0.3891
Best Hyperparameters: {'n_estimators': 105, 'max_depth': 12, 'min_samples_split': 0.21006891602905914, 'min_samples_leaf': 0.1596071415236025}
